In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "examples" else CWD
REF_EXAMPLES = ROOT.parent / "ref" / "CC-FJpy-master" / "examples"
sys.path.insert(0, str(ROOT))

import ccfj


In [ ]:
# 纯 Python 版本计算较慢，这里使用经过验证的较小数据规模。
data = np.load(REF_EXAMPLES / "summed.npz")
ncfs = data["ncfs"][:16, :128]
r = data["r"][:16] * 1e3
f = data["f"][:128]
c = np.linspace(2000, 5000, 180, dtype=np.float32)

print({
    "ncfs_shape": ncfs.shape,
    "r_shape": r.shape,
    "f_shape": f.shape,
    "c_shape": c.shape,
})


In [ ]:
cases = [
    ("fj_noise_j_trap", dict(fstride=1, itype=0, func=0)),
    ("fj_noise_j_int", dict(fstride=1, itype=1, func=0)),
    ("fj_noise_y_trap", dict(fstride=1, itype=0, func=1)),
    ("fj_noise_y_int", dict(fstride=1, itype=1, func=1)),
]

results = {}
for name, kwargs in cases:
    ds = ccfj.fj_noise(np.real(ncfs), r, c, f, **kwargs)
    results[name] = ds

summary = {
    name: {
        "shape": list(ds.shape),
        "max": float(np.max(ds)),
        "min": float(np.min(ds)),
        "mean": float(np.mean(ds)),
    }
    for name, ds in results.items()
}
summary


In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 12), constrained_layout=True)
for idx, (name, ds) in enumerate(results.items()):
    row, col = divmod(idx, 2)
    im = ax[row][col].pcolormesh(f, c / 1e3, ds, cmap="jet", vmin=0, vmax=0.8, shading="auto")
    ax[row][col].set_xlim([0, 0.5])
    ax[row][col].set_xlabel("Frequency (Hz)")
    ax[row][col].set_ylabel("Phase velocity (km/s)")
    ax[row][col].set_title(name)
    fig.colorbar(im, ax=ax[row][col], fraction=0.046, pad=0.04)
plt.show()
print("中文自检：噪声 F-J notebook 已按 UTF-8 写入。")
